# ベイズ最適化

合成関数を対象に、GPとEIによる逐次探索を実装する。ランダム探索と同じ初期点・観測回数で比較する。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from bayesian_optimization import fit_precomputed_gp, expected_improvement, run_search

X = np.linspace(0, 1, 201).reshape(-1, 1)
# 後ろ向き評価用の合成目的関数。選択ロジックには観測済みの値のみを渡す。
y = np.sin(9 * X[:, 0]) + 0.3 * np.cos(23 * X[:, 0]) + 0.5 * X[:, 0]
length_scale = 0.08
kernel = np.exp(-0.5 * cdist(X, X, metric="sqeuclidean") / length_scale**2)


## 1. GP予測とEI

観測値を標準化し、固定カーネルで予測平均・標準偏差を計算する。EIが最大の未観測点を次の候補とする。


In [ ]:
initial = np.array([10, 80, 150])
candidates = np.setdiff1d(np.arange(len(y)), initial)
model = fit_precomputed_gp(kernel, y, initial)
mean, std = model.predict(candidates)
ei = expected_improvement(mean, std, y[initial].max())
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(X[:, 0], y, "--", alpha=0.4, label="Synthetic objective (evaluation only)")
axes[0].plot(X[candidates, 0], mean, label="GP mean")
axes[0].fill_between(X[candidates, 0], mean-1.96*std, mean+1.96*std, alpha=0.2)
axes[0].scatter(X[initial, 0], y[initial], color="black", label="Observed")
axes[0].legend()
axes[1].plot(X[candidates, 0], ei, label="Expected improvement")
axes[1].set_xlabel("x")
axes[1].legend()
plt.tight_layout()
plt.show()


## 2. ランダム探索との比較

初期点3点、追加観測20回、10 seedで比較する。評価はsimple regretの平均と標準偏差を用いる。


In [ ]:
records = []
for seed in range(10):
    initial = np.random.default_rng(seed).choice(len(y), size=3, replace=False)
    for method in ("Random", "BO"):
        selected, best = run_search(kernel, y, initial, n_steps=20, method=method, seed=seed)
        assert len(np.unique(selected)) == len(selected)
        assert np.all(np.diff(best) >= -1e-12)
        for step, value in enumerate(best):
            records.append(dict(seed=seed, method=method, observations=3+step,
                                best=value, simple_regret=y.max()-value))
results = pd.DataFrame(records)
fig, ax = plt.subplots(figsize=(8, 4))
for method, rows in results.groupby("method"):
    summary = rows.groupby("observations")["simple_regret"].agg(["mean", "std"])
    ax.plot(summary.index, summary["mean"], label=method)
    ax.fill_between(summary.index, np.maximum(0, summary["mean"]-summary["std"]),
                    summary["mean"]+summary["std"], alpha=0.15)
ax.set(xlabel="Total observations", ylabel="Simple regret",
       title="Mean ± standard deviation across 10 seeds")
ax.legend()
plt.tight_layout()
plt.show()
results.groupby(["method", "observations"])["simple_regret"].mean().unstack(0).tail()


## 制約

- 有限候補集合・単一目的の最大化。
- カーネルとノイズ分散は固定。
- 合成関数上の比較であり、実データでの有効性は未検証。
